# Step 02 — Star Schema

Takes the raw text tables from Step 01 and builds a typed dimensional model: 6 dimensions,
1 bridge, 3 facts. Every cast is explicit, and every build is followed by an assertion so a
silent fan-out or a dropped row fails here rather than showing up as a wrong number on a
dashboard three weeks from now.

**Design decisions, and why:**

| Decision | Rationale |
|---|---|
| Junk dimension for `type` × `operation` × `k_symbol` | 3 text columns on 1.06M rows collapse to one integer key. Only 15 distinct combinations exist |
| Bridge for client↔account | Step 01 confirmed clients hold multiple accounts and accounts have a second `DISPONENT` user. A direct relationship would be wrong |
| `dim_district` kept flat, not snowflaked | Region is one column of 77 rows. Snowflaking buys nothing and costs a join |
| Czech codes translated, raw code retained | Dashboard reads in English, traceability back to source preserved |
| `date_key` as `YYYYMMDD` integer | Smallest possible key on the large fact, and human-readable when debugging |
| Signed amount stored, not computed in DAX | The sign rule is a data-quality decision settled in Step 01. It belongs in the pipeline, not in a measure |


## 2.0 Setup


In [1]:
!pip install -q duckdb


In [2]:
import duckdb, pandas as pd, shutil, os
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/berka-banking-analytics'
PQ   = f'{BASE}/Data/parquet'
OUT  = f'{BASE}/Step02_Model'
SHARED_DB = f'{BASE}/Data/berka.duckdb'
DB   = '/content/berka.duckdb'

os.makedirs(PQ, exist_ok=True); os.makedirs(OUT, exist_ok=True)
shutil.copy(SHARED_DB, DB)
con = duckdb.connect(DB)
print(sorted(con.sql("SELECT table_name FROM duckdb_tables()").df().table_name))

Mounted at /content/drive
['raw_account', 'raw_card', 'raw_client', 'raw_disp', 'raw_district', 'raw_loan', 'raw_order', 'raw_trans']


## 2.1 `dim_date`

Built as a continuous spine from `generate_series`, not from the distinct dates present in the
facts. A gap-free calendar is what makes DAX time intelligence work; a date table assembled from
observed dates breaks `DATEADD` on any month with no transactions.

Range is 1993-01-01 → 1998-12-31, taken from the Step 01 date audit.


In [4]:
con.execute("""
CREATE OR REPLACE TABLE dim_date AS
SELECT CAST(strftime(d,'%Y%m%d') AS INTEGER) date_key, d AS full_date,
       year(d) calendar_year, quarter(d) calendar_quarter, month(d) calendar_month,
       monthname(d) month_name, CAST(strftime(d,'%Y%m') AS INTEGER) year_month,
       year(d)||'-Q'||quarter(d) year_quarter, day(d) day_of_month,
       dayname(d) day_name, isodow(d) day_of_week, isodow(d)>=6 is_weekend,
       date_trunc('month',d)::DATE month_start
FROM generate_series(DATE '1993-01-01', DATE '1998-12-31', INTERVAL 1 DAY) t(d)
""")
con.sql("SELECT count(*) n_days, min(full_date) lo, max(full_date) hi FROM dim_date").df()

,n_days,lo,hi
0,2191,1993-01-01,1998-12-31


## 2.2 `dim_district`

`A1`–`A16` renamed to meaning. `A12` and `A15` use `TRY_CAST` because of the single `?` row
found in Step 01 — the district is kept, only those two measures go NULL.


In [5]:
con.execute("""
CREATE OR REPLACE TABLE dim_district AS
SELECT CAST(A1 AS INTEGER) district_key, A2 district_name, A3 region,
       CAST(A4 AS INTEGER) n_inhabitants,
       CAST(A5 AS INTEGER) n_munic_lt499, CAST(A6 AS INTEGER) n_munic_500_1999,
       CAST(A7 AS INTEGER) n_munic_2000_9999, CAST(A8 AS INTEGER) n_munic_gt10000,
       CAST(A9 AS INTEGER) n_cities, CAST(A10 AS DOUBLE) pct_urban,
       CAST(A11 AS INTEGER) avg_salary,
       TRY_CAST(A12 AS DOUBLE) unemployment_95, CAST(A13 AS DOUBLE) unemployment_96,
       CAST(A14 AS INTEGER) entrepreneurs_per_1000,
       TRY_CAST(A15 AS INTEGER) crimes_95, CAST(A16 AS INTEGER) crimes_96
FROM raw_district
""")
con.sql("SELECT district_key, district_name, unemployment_95, crimes_95 FROM dim_district WHERE unemployment_95 IS NULL OR crimes_95 IS NULL").df()

,district_key,district_name,unemployment_95,crimes_95
0,69,Jesenik,NaN,<NA>


## 2.3 `dim_client`

`birth_number` decoded to date of birth and gender. Age is computed as at 1998-12-31, the last
date in the data, so it is fixed and reproducible rather than drifting with today's date.

Age bands are 10-year groups. This is a presentation convenience, not a modelling threshold —
`age_at_1998` is kept alongside so nothing is thrown away.


In [6]:
con.execute("""
CREATE OR REPLACE TABLE dim_client AS
WITH d AS (SELECT CAST(client_id AS INTEGER) client_key, CAST(district_id AS INTEGER) district_key,
                  CAST(substr(birth_number,1,2) AS INT) yy,
                  CAST(substr(birth_number,3,2) AS INT) mm,
                  CAST(substr(birth_number,5,2) AS INT) dd FROM raw_client),
     b AS (SELECT client_key, district_key,
                  CASE WHEN mm > 50 THEN 'Female' ELSE 'Male' END gender,
                  make_date(1900+yy, CASE WHEN mm > 50 THEN mm-50 ELSE mm END, dd) birth_date
           FROM d)
SELECT client_key, district_key, gender, birth_date,
       CAST(date_diff('year', birth_date, DATE '1998-12-31') AS INTEGER) age_at_1998,
       CASE WHEN date_diff('year', birth_date, DATE '1998-12-31') < 25 THEN '<25'
            WHEN date_diff('year', birth_date, DATE '1998-12-31') < 35 THEN '25-34'
            WHEN date_diff('year', birth_date, DATE '1998-12-31') < 45 THEN '35-44'
            WHEN date_diff('year', birth_date, DATE '1998-12-31') < 55 THEN '45-54'
            WHEN date_diff('year', birth_date, DATE '1998-12-31') < 65 THEN '55-64'
            ELSE '65+' END age_band
FROM b
""")
con.sql("SELECT gender, count(*) n, min(age_at_1998) youngest, max(age_at_1998) oldest FROM dim_client GROUP BY 1").df()


,gender,n,youngest,oldest
0,Female,2645,11,84
1,Male,2724,12,87


## 2.4 `dim_account` and `dim_card`


In [7]:
con.execute("""
CREATE OR REPLACE TABLE dim_account AS
SELECT CAST(account_id AS INTEGER) account_key, CAST(district_id AS INTEGER) district_key,
       frequency frequency_cz,
       CASE frequency WHEN 'POPLATEK MESICNE'   THEN 'Monthly'
                      WHEN 'POPLATEK TYDNE'     THEN 'Weekly'
                      WHEN 'POPLATEK PO OBRATU' THEN 'After transaction' END statement_frequency,
       strptime(date,'%y%m%d')::DATE opened_date,
       CAST(strftime(strptime(date,'%y%m%d'),'%Y%m%d') AS INTEGER) opened_date_key
FROM raw_account
""")

con.execute("""
CREATE OR REPLACE TABLE dim_card AS
SELECT CAST(card_id AS INTEGER) card_key, CAST(disp_id AS INTEGER) disp_key,
       upper(type[1]) || type[2:] card_type,
       strptime(issued,'%y%m%d %H:%M:%S')::DATE issued_date,
       CAST(strftime(strptime(issued,'%y%m%d %H:%M:%S'),'%Y%m%d') AS INTEGER) issued_date_key
FROM raw_card
""")
con.sql("SELECT statement_frequency, count(*) n FROM dim_account GROUP BY 1 ORDER BY 2 DESC").df()


,statement_frequency,n
0,Monthly,4167
1,Weekly,240
2,After transaction,93


## 2.5 `dim_transaction_type` — the junk dimension

This is the single biggest model-size win. `type`, `operation` and `k_symbol` are three
low-cardinality text columns sitting on a 1.06M-row fact. Every distinct combination is
enumerated once, given a surrogate key, and the fact carries the integer instead.

`IS NOT DISTINCT FROM` is used for the join back, not `=`, because both `operation` and
`k_symbol` contain NULLs and `NULL = NULL` is never true. Using `=` here would silently drop
every interest transaction.


In [8]:
con.execute("""
CREATE OR REPLACE TABLE dim_transaction_type AS
WITH combos AS (SELECT DISTINCT type type_cz, operation operation_cz,
                       nullif(trim(k_symbol),'') k_symbol_cz FROM raw_trans)
SELECT row_number() OVER (ORDER BY type_cz, operation_cz NULLS FIRST, k_symbol_cz NULLS FIRST) trans_type_key,
       type_cz, operation_cz, k_symbol_cz,
       CASE type_cz WHEN 'PRIJEM' THEN 'Credit' ELSE 'Debit' END direction,
       type_cz = 'PRIJEM' is_credit,
       CASE operation_cz WHEN 'VKLAD'          THEN 'Cash deposit'
                         WHEN 'VYBER'          THEN 'Cash withdrawal'
                         WHEN 'PREVOD Z UCTU'  THEN 'Collection from another bank'
                         WHEN 'PREVOD NA UCET' THEN 'Remittance to another bank'
                         WHEN 'VYBER KARTOU'   THEN 'Credit card withdrawal'
                         ELSE 'Interest credited' END operation_en,
       CASE k_symbol_cz WHEN 'POJISTNE'    THEN 'Insurance payment'
                        WHEN 'SLUZBY'      THEN 'Statement charge'
                        WHEN 'UROK'        THEN 'Interest credited'
                        WHEN 'SANKC. UROK' THEN 'Sanction interest'
                        WHEN 'SIPO'        THEN 'Household payment'
                        WHEN 'DUCHOD'      THEN 'Old-age pension'
                        WHEN 'UVER'        THEN 'Loan payment'
                        ELSE 'Unknown' END purpose_en
FROM combos
""")
con.sql("SELECT trans_type_key, direction, operation_en, purpose_en FROM dim_transaction_type ORDER BY 1").df()


,trans_type_key,direction,operation_en,purpose_en
0,1,Credit,Interest credited,Interest credited
1,2,Credit,Collection from another bank,Unknown
2,3,Credit,Collection from another bank,Old-age pension
3,4,Credit,Cash deposit,Unknown
4,5,Debit,Cash withdrawal,Unknown
5,6,Debit,Remittance to another bank,Unknown
6,7,Debit,Remittance to another bank,Insurance payment
7,8,Debit,Remittance to another bank,Household payment
8,9,Debit,Remittance to another bank,Loan payment
9,10,Debit,Cash withdrawal,Unknown


## 2.6 `bridge_disposition`

The many-to-many resolver. Step 01 established one `OWNER` per account plus 869 `DISPONENT`
rows, with clients appearing on more than one account, so this relationship cannot be flattened
onto either dimension without either losing rows or double-counting.


In [9]:
con.execute("""
CREATE OR REPLACE TABLE bridge_disposition AS
SELECT CAST(disp_id AS INTEGER) disp_key, CAST(client_id AS INTEGER) client_key,
       CAST(account_id AS INTEGER) account_key,
       upper(type[1]) || lower(type[2:]) disposition_role, type = 'OWNER' is_owner
FROM raw_disp
""")
con.sql("SELECT disposition_role, count(*) n FROM bridge_disposition GROUP BY 1").df()


,disposition_role,n
0,Owner,4500
1,Disponent,869


## 2.7 Facts

`signed_amount` applies the sign rule settled empirically in Step 01: `PRIJEM` is a credit,
everything else is a debit, including the 16,666 rows mis-coded `VYBER` in the `type` column.
`balance` is carried through as recorded and never recomputed.


In [10]:
con.execute("""
CREATE OR REPLACE TABLE fact_transaction AS
SELECT CAST(t.trans_id AS BIGINT) trans_id,
       CAST(t.account_id AS INTEGER) account_key,
       CAST(strftime(strptime(t.date,'%y%m%d'),'%Y%m%d') AS INTEGER) date_key,
       tt.trans_type_key,
       CAST(t.amount AS DECIMAL(12,2)) amount,
       CAST(CASE WHEN t.type='PRIJEM' THEN CAST(t.amount AS DOUBLE)
                 ELSE -CAST(t.amount AS DOUBLE) END AS DECIMAL(12,2)) signed_amount,
       CAST(t.balance AS DECIMAL(12,2)) balance
FROM raw_trans t
JOIN dim_transaction_type tt
  ON t.type = tt.type_cz
 AND t.operation IS NOT DISTINCT FROM tt.operation_cz
 AND nullif(trim(t.k_symbol),'') IS NOT DISTINCT FROM tt.k_symbol_cz
""")

con.execute("""
CREATE OR REPLACE TABLE fact_loan AS
SELECT CAST(loan_id AS INTEGER) loan_id, CAST(account_id AS INTEGER) account_key,
       CAST(strftime(strptime(date,'%y%m%d'),'%Y%m%d') AS INTEGER) date_key,
       CAST(amount AS DECIMAL(12,2)) loan_amount, CAST(duration AS INTEGER) duration_months,
       CAST(payments AS DECIMAL(12,2)) monthly_payment, status status_code,
       CASE status WHEN 'A' THEN 'Finished, paid'  WHEN 'B' THEN 'Finished, defaulted'
                   WHEN 'C' THEN 'Running, OK'     WHEN 'D' THEN 'Running, in debt' END status_desc,
       status IN ('A','B') is_finished, status IN ('B','D') is_default
FROM raw_loan
""")

con.execute("""
CREATE OR REPLACE TABLE fact_order AS
SELECT CAST(order_id AS INTEGER) order_id, CAST(account_id AS INTEGER) account_key,
       bank_to, account_to, CAST(amount AS DECIMAL(12,2)) amount,
       nullif(trim(k_symbol),'') k_symbol_cz,
       CASE nullif(trim(k_symbol),'') WHEN 'POJISTNE' THEN 'Insurance payment'
            WHEN 'SIPO'    THEN 'Household payment' WHEN 'LEASING' THEN 'Leasing'
            WHEN 'UVER'    THEN 'Loan payment'      ELSE 'Unknown' END purpose_en
FROM raw_order
""")
print('facts built')


facts built


## 2.8 Integrity assertions

The point of this cell is to fail. A join that fans out, a key that goes NULL, or a total that
shifts is caught here where the cause is one cell away, instead of in Power BI where it looks
like a DAX problem.


In [11]:
raw_n  = con.sql('SELECT count(*) FROM raw_trans').fetchone()[0]
fact_n = con.sql('SELECT count(*) FROM fact_transaction').fetchone()[0]
assert raw_n == fact_n, f'fan-out or loss: {raw_n} -> {fact_n}'

nulls = con.sql("""SELECT count(*) FILTER (WHERE account_key IS NULL)
                        + count(*) FILTER (WHERE date_key IS NULL)
                        + count(*) FILTER (WHERE trans_type_key IS NULL) FROM fact_transaction""").fetchone()[0]
assert nulls == 0, f'{nulls} null foreign keys'

raw_sum  = con.sql('SELECT sum(CAST(amount AS DOUBLE)) FROM raw_trans').fetchone()[0]
fact_sum = float(con.sql('SELECT sum(amount) FROM fact_transaction').fetchone()[0])
assert abs(raw_sum - fact_sum) < 1, f'amount total moved by {raw_sum - fact_sum}'

print(f'rows {raw_n:,} preserved | null FKs 0 | amount total {fact_sum:,.2f} (delta {raw_sum-fact_sum:.2f})')

rows 1,056,320 preserved | null FKs 0 | amount total 6,257,793,560.30 (delta -0.01)


In [12]:
con.sql("""
SELECT 'fact_transaction.date_key -> dim_date' rel, count(*) orphans
  FROM fact_transaction f LEFT JOIN dim_date d USING(date_key) WHERE d.date_key IS NULL
UNION ALL SELECT 'fact_transaction.account_key -> dim_account', count(*)
  FROM fact_transaction f LEFT JOIN dim_account a USING(account_key) WHERE a.account_key IS NULL
UNION ALL SELECT 'fact_transaction.trans_type_key -> dim_transaction_type', count(*)
  FROM fact_transaction f LEFT JOIN dim_transaction_type t USING(trans_type_key) WHERE t.trans_type_key IS NULL
UNION ALL SELECT 'fact_loan.date_key -> dim_date', count(*)
  FROM fact_loan f LEFT JOIN dim_date d USING(date_key) WHERE d.date_key IS NULL
UNION ALL SELECT 'fact_order.account_key -> dim_account', count(*)
  FROM fact_order f LEFT JOIN dim_account a USING(account_key) WHERE a.account_key IS NULL
UNION ALL SELECT 'dim_account.opened_date_key -> dim_date', count(*)
  FROM dim_account a LEFT JOIN dim_date d ON a.opened_date_key = d.date_key WHERE d.date_key IS NULL
UNION ALL SELECT 'dim_card.issued_date_key -> dim_date', count(*)
  FROM dim_card c LEFT JOIN dim_date d ON c.issued_date_key = d.date_key WHERE d.date_key IS NULL
UNION ALL SELECT 'dim_card.disp_key -> bridge', count(*)
  FROM dim_card c LEFT JOIN bridge_disposition b USING(disp_key) WHERE b.disp_key IS NULL
UNION ALL SELECT 'dim_client.district_key -> dim_district', count(*)
  FROM dim_client c LEFT JOIN dim_district d USING(district_key) WHERE d.district_key IS NULL
UNION ALL SELECT 'dim_account.district_key -> dim_district', count(*)
  FROM dim_account a LEFT JOIN dim_district d USING(district_key) WHERE a.district_key IS NOT NULL AND d.district_key IS NULL
""").df()

,rel,orphans
0,fact_transaction.date_key -> dim_date,0
1,fact_transaction.account_key -> dim_account,0
2,fact_transaction.trans_type_key -> dim_transac...,0
3,fact_loan.date_key -> dim_date,0
4,fact_order.account_key -> dim_account,0
5,dim_account.opened_date_key -> dim_date,0
6,dim_card.issued_date_key -> dim_date,0
7,dim_card.disp_key -> bridge,0
8,dim_client.district_key -> dim_district,0
9,dim_account.district_key -> dim_district,0


## 2.9 Export to Parquet

Parquet rather than CSV: types survive the handoff, so Power Query has nothing to infer and
nothing to get wrong. ZSTD compression brings the whole model well under the raw text size.


In [13]:
MODEL = ['dim_date','dim_district','dim_client','dim_account','dim_card',
         'dim_transaction_type','bridge_disposition','fact_transaction','fact_loan','fact_order']

for t in MODEL:
    con.execute(f"COPY {t} TO '{PQ}/{t}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)")

sizes = pd.DataFrame([(t, os.path.getsize(f'{PQ}/{t}.parquet')) for t in MODEL], columns=['table','bytes'])
sizes['mb'] = (sizes.bytes/1024**2).round(2)
print(f"total {sizes.bytes.sum()/1024**2:.1f} MB")
sizes.sort_values('bytes', ascending=False)


total 13.1 MB


,table,bytes,mb
7,fact_transaction,13313353,12.70
9,fact_order,112722,0.11
6,bridge_disposition,86955,0.08
2,dim_client,72146,0.07
3,dim_account,53852,0.05
0,dim_date,33560,0.03
8,fact_loan,21005,0.02
4,dim_card,18605,0.02
1,dim_district,8585,0.01
5,dim_transaction_type,2277,0.00


In [14]:
sizes.to_csv(f'{OUT}/model_table_sizes.csv', index=False)
con.close()
shutil.copy(DB, SHARED_DB)
print('model written back →', SHARED_DB)
print(os.listdir(OUT))

model written back → /content/drive/MyDrive/berka-banking-analytics/Data/berka.duckdb
['model_table_sizes.csv', 'berka_model.duckdb']


---

## Model shape

```
                  dim_date
                     |  (1 active + 3 inactive: loan, opened, issued)
dim_district --- dim_account --- fact_transaction --- dim_transaction_type
      |               |    \
      |               |     +-- fact_loan
      |               |     +-- fact_order
      |               |
dim_client -- bridge_disposition -- dim_card
```

## Carried into Step 03

- `dim_date` connects to `fact_transaction.date_key` as the **active** relationship. `fact_loan.date_key`,
  `dim_account.opened_date_key` and `dim_card.issued_date_key` become **inactive** relationships,
  activated per measure with `USERELATIONSHIP`. Do not duplicate the date table.
- `dim_district` is reachable from both `dim_account` and `dim_client`. Make the account path active,
  the client path inactive, and expose the switch through a measure.
- `bridge_disposition` stays single-direction. Use `CROSSFILTER` inside the specific measures that
  need client-level filtering rather than switching the relationship to both-directional.
- `fact_loan` is 1:1 with account, so loan measures and account measures must not be summed together.
- Step 03 builds the monthly balance snapshot on `fact_transaction.balance`, the recorded value,
  plus loan vintages and the account behaviour features.
